# Power Breakers

**Name(s)**: 
Alex Twoy <atwoy@ucsd.edu> - A17580384
Mitchell Farrington <mfarring@ucsd.edu>  - A15378625
Anuradha Jaganathan <ajaganathan@ucsd.edu> - A69038119
Kyle Packer <kpacker@ucsd.edu> - A69036932

**Website Link**: (your website link)

In [121]:
import pandas as pd
import numpy as np
from pathlib import Path


import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc259r_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

In [102]:
# TODO

## Step 2: Data Cleaning and Exploratory Data Analysis

In [103]:
import sys                                                                                                            
!uv pip install openpyxl  
!uv pip install jupyterlab plotly


Using Python 3.12.12 environment at: /Users/ajagan/Documents/Q12026_UCSD/dsc259r-2026-wi/.venv
Audited 1 package in 5ms
Using Python 3.12.12 environment at: /Users/ajagan/Documents/Q12026_UCSD/dsc259r-2026-wi/.venv
Audited 2 packages in 42ms


In [104]:
outage_url = 'https://engineering.purdue.edu/LASCI/research-data/outages/outage.xlsx'
df = pd.read_excel(outage_url, skiprows=list(range(5))+[6], index_col = 'OBS')
df.head()

,variables,YEAR,MONTH,U.S._STATE,...,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
OBS,,,,,,,,,
1,NaN,2011,7.0,Minnesota,...,0.6,91.59,8.41,5.48
2,NaN,2014,5.0,Minnesota,...,0.6,91.59,8.41,5.48
3,NaN,2010,10.0,Minnesota,...,0.6,91.59,8.41,5.48
4,NaN,2012,6.0,Minnesota,...,0.6,91.59,8.41,5.48
5,NaN,2015,7.0,Minnesota,...,0.6,91.59,8.41,5.48


In [105]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1534 entries, 1 to 1534
Data columns (total 56 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   variables                0 non-null      float64       
 1   YEAR                     1534 non-null   int64         
 2   MONTH                    1525 non-null   float64       
 3   U.S._STATE               1534 non-null   object        
 4   POSTAL.CODE              1534 non-null   object        
 5   NERC.REGION              1534 non-null   object        
 6   CLIMATE.REGION           1528 non-null   object        
 7   ANOMALY.LEVEL            1525 non-null   float64       
 8   CLIMATE.CATEGORY         1525 non-null   object        
 9   OUTAGE.START.DATE        1525 non-null   datetime64[ns]
 10  OUTAGE.START.TIME        1525 non-null   object        
 11  OUTAGE.RESTORATION.DATE  1476 non-null   datetime64[ns]
 12  OUTAGE.RESTORATION.TIME  1476 non-null 

In [106]:
outage_df = df.copy()
outage_df = outage_df.drop(columns=['variables', 'RES.PRICE','COM.PRICE', 'IND.PRICE','TOTAL.PRICE', 
                                    'RES.PRICE','RES.SALES', 'COM.SALES','IND.SALES', 'TOTAL.SALES','PC.REALGSP.STATE', 'PC.REALGSP.USA','PC.REALGSP.REL', 'PC.REALGSP.CHANGE','UTIL.REALGSP', 'TOTAL.REALGSP','UTIL.CONTRI', 'PI.UTIL.OFUSA'])

outage_df.columns = outage_df.columns.str.replace('.', '_', regex=False)                                              

outage_df = outage_df.iloc[1:].reset_index(drop=True)


outage_df = outage_df.rename(columns={'U_S__STATE': 'US_STATE', 'POSTAL_CODE': 'STATE_CODE', 'OUTAGE_START_DATE(Day of the week, Month Day, Year)':'OUTAGE_START_DATE',
                                        'OUTAGE_START_TIME(Hour:Minute:Second (AM / PM))': 'OUTAGE_START_TIME', 'OUTAGE_RESTORATION_DATE(Day of the week, Month Day, Year)': 'OUTAGE_RESTORATION_DATE',
                                        'OUTAGE_RESTORATION_TIME(Hour:Minute:Second (AM / PM))':'OUTAGE_RESTORATION_TIME', 'POPDEN_URBAN(persons per square mile)':'POP_DENSITY_URBAN',
                                        'POPDEN_UC(persons per square mile)':'POP_DENSITY_UC','POPDEN_RURAL(persons per square mile)':'POP_DENSITY_RURAL',
                                        'OUTAGE_RESTORATION_TIME(Hour:Minute:Second (AM / PM))':'OUTAGE_RESTORATION_TIME','RES_PERCEN':'RES_PERCENT','COM_PERCEN':'COM_PERCENT','IND_PERCEN':'IND_PERCENT',
                                        'RES_CUST_PCT':'RES_CUST_PERCENT','COM_CUST_PCT':'COM_CUST_PERCENT','IND_CUST_PCT':'IND_CUST_PERCENT','PCT_LAND':'LAND_AREA_PERCENT','PCT_WATER_TOT':'TOTAL_WATER_AREA_PERCENT',
                                        'PCT_WATER_INLAND': 'INLAND_WATER_AREA_PERCENT','AREAPCT_URBAN':'URBAN_AREA_PERCENT','AREAPCT_UC':'UC_AREA_PERCENT','OUTAGE_DURATION':'OUTAGE_DURATION_MINS'
                                     }) 

outage_df['OUTAGE_START_TIMESTAMP'] = pd.to_datetime(                                                                 
      outage_df['OUTAGE_START_DATE'].astype(str) + ' ' + outage_df['OUTAGE_START_TIME'].astype(str),
      format='%Y-%m-%d %H:%M:%S',                                                                                       
      errors='coerce'
  )

outage_df['OUTAGE_RESTORATION_TIMESTAMP'] = pd.to_datetime(                                                                 
      outage_df['OUTAGE_RESTORATION_DATE'].astype(str) + ' ' + outage_df['OUTAGE_RESTORATION_TIME'].astype(str),
      format='%Y-%m-%d %H:%M:%S',                                                                                       
      errors='coerce'
  )

outage_df = outage_df.drop(columns=['OUTAGE_START_DATE','OUTAGE_START_TIME','OUTAGE_RESTORATION_DATE','OUTAGE_RESTORATION_TIME'])

intcols = ['MONTH','OUTAGE_DURATION_MINS','CUSTOMERS_AFFECTED','DEMAND_LOSS_MW','POPULATION']
outage_df[intcols] = outage_df[intcols].apply(pd.to_numeric, errors='coerce').astype('Int64')

outage_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1533 entries, 0 to 1532
Data columns (total 37 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   YEAR                          1533 non-null   int64         
 1   MONTH                         1524 non-null   Int64         
 2   US_STATE                      1533 non-null   object        
 3   STATE_CODE                    1533 non-null   object        
 4   NERC_REGION                   1533 non-null   object        
 5   CLIMATE_REGION                1527 non-null   object        
 6   ANOMALY_LEVEL                 1524 non-null   float64       
 7   CLIMATE_CATEGORY              1524 non-null   object        
 8   CAUSE_CATEGORY                1533 non-null   object        
 9   CAUSE_CATEGORY_DETAIL         1063 non-null   object        
 10  HURRICANE_NAMES               72 non-null     object        
 11  OUTAGE_DURATION_MINS          

In [107]:
display_df(outage_df.sort_values('OUTAGE_START_TIMESTAMP'),
           rows=5)

,YEAR,MONTH,US_STATE,STATE_CODE,...,TOTAL_WATER_AREA_PERCENT,INLAND_WATER_AREA_PERCENT,OUTAGE_START_TIMESTAMP,OUTAGE_RESTORATION_TIMESTAMP
809,2000,1,South Carolina,SC,...,6.12,3.32,2000-01-23 08:00:00,2000-01-28 12:00:00
811,2000,1,South Carolina,SC,...,6.12,3.32,2000-01-29 22:00:00,2000-02-03 12:00:00
...,...,...,...,...,...,...,...,...,...
1529,2006,<NA>,North Dakota,ND,...,2.40,2.40,NaT,NaT
1532,2000,<NA>,Alaska,AK,...,14.24,2.90,NaT,NaT


In [108]:
outage_df.head()

,YEAR,MONTH,US_STATE,STATE_CODE,...,TOTAL_WATER_AREA_PERCENT,INLAND_WATER_AREA_PERCENT,OUTAGE_START_TIMESTAMP,OUTAGE_RESTORATION_TIMESTAMP
0,2014,5,Minnesota,MN,...,8.41,5.48,2014-05-11 18:38:00,2014-05-11 18:39:00
1,2010,10,Minnesota,MN,...,8.41,5.48,2010-10-26 20:00:00,2010-10-28 22:00:00
2,2012,6,Minnesota,MN,...,8.41,5.48,2012-06-19 04:30:00,2012-06-20 23:00:00
3,2015,7,Minnesota,MN,...,8.41,5.48,2015-07-18 02:00:00,2015-07-19 07:00:00
4,2010,11,Minnesota,MN,...,8.41,5.48,2010-11-13 15:00:00,2010-11-14 22:00:00


In [109]:
outage_df = outage_df.dropna(subset=['OUTAGE_DURATION_MINS'])                                                            
                                                                                                                        
bins = [0, 500, 1000, 5000, 10000, 15000, 110000]
outage_df['duration_bins']

KeyError: 'duration_bins'

In [110]:
#States with more outage
states_max_outage = outage_df.groupby('STATE_CODE')['OUTAGE_DURATION_MINS'].sum().reset_index()
states_max_outage.columns = ['state', 'count']
#display_df(states_max_outage.sort_values(ascending=False),rows=50)
fig = px.choropleth(states_max_outage,                                                                                       
                      locations='state',
                      locationmode='USA-states',
                      color='count',
                      scope='usa',
                      color_continuous_scale='Reds',
                      title='Number of Outages by State')
fig.show(renderer='iframe')

In [111]:
all_states=outage_df['STATE_CODE'].value_counts().reset_index()
all_states.columns = ['state', 'count']
#display_df(states_max_outage.sort_values(ascending=False),rows=50)
                                                                                                                        
fig = px.choropleth(all_states,                                                                                       
                      locations='state',
                      locationmode='USA-states',
                      color='count',
                      scope='usa',
                      color_continuous_scale='Reds',
                      title='Number of Outages by State')
fig.show(renderer='iframe')

In [112]:
outage_df['duration_bins'].isna().value_counts()
fig = px.histogram(df_clean, x="OUTAGE_DURATION_MINS", nbins=100,
                     title="Distribution of Outage Duration",
                     labels={"OUTAGE_DURATION_MINS": "Duration (mins)"})
fig.show(renderer='iframe')

KeyError: 'duration_bins'

In [113]:
filtered = outage_df.dropna(subset=['OUTAGE_DURATION_MINS'])

fig = px.box(filtered, x='CLIMATE_CATEGORY', y='OUTAGE_DURATION_MINS',
               title='Outage Duration by Cause Category (< 5000 mins)')
fig.show(renderer='iframe')

In [114]:
filtered = outage_df.dropna(subset=['OUTAGE_DURATION_MINS'])

fig = px.box(filtered, x='CAUSE_CATEGORY', y='OUTAGE_DURATION_MINS',
               title='Outage Duration by Cause Category (< 5000 mins)')
fig.show(renderer='iframe')

In [115]:
outage_by_region_cause = outage_df.groupby(['CLIMATE_REGION',                                                         
'CAUSE_CATEGORY'])['OUTAGE_DURATION_MINS'].sum().reset_index()                                                        
outage_by_region_cause   
fig = px.bar(outage_by_region_cause,
               x='CLIMATE_REGION',                                                                                      
               y='OUTAGE_DURATION_MIN',                                                                                
               color='CAUSE_CATEGORY',
               title='Outage Duration by Region and Cause',
               labels={'OUTAGE_DURATION_MINS': 'Total Duration (mins)',
                       'CLIMATE_REGION': 'Region'})
fig.show(renderer='iframe')

In [129]:
  filtered = outage_df.dropna(subset=['OUTAGE_DURATION_MINS']).copy()                                                   
  filtered['OUTAGE_DURATION_MINS'] = filtered['OUTAGE_DURATION_MINS'].astype(float)
  filtered['has_hurricane'] = filtered['HURRICANE_NAMES'].notna()                                                       
                  
  fig = create_kde_plotly(
      filtered,
      'has_hurricane',
      True,
      False,
      'OUTAGE_DURATION_MINS',
      title='Distribution of Outage Duration<br>Based on Hurricane Involvement'
  )
  fig.show(renderer='iframe')

In [150]:
cause_category_climate = outage_df.pivot_table(
                    index='CAUSE_CATEGORY', 
                    columns='CLIMATE_CATEGORY',
                    values='OUTAGE_DURATION_MINS',
                    aggfunc='sum',
                    fill_value=0
                )
cause_category_climate.assign(total=cause_category_climate.sum(axis=1)).sort_values('total', ascending=False).drop(columns='total')
display_df(cause_category_climate,
           rows=7)

CLIMATE_CATEGORY,cold,normal,warm
CAUSE_CATEGORY,,,
equipment failure,5240,89640,5050
fuel supply emergency,313794,130200,68399
intentional attack,58182,96034,19066
islanding,3889,2417,2518
public appeal,46770,46802,7751
severe weather,780629,1425825,680171
system operability disruption,21667,53638,14346


In [136]:
#Aggregates
prodname = outage_df.pivot_table(
                    index='CAUSE_CATEGORY', 
                    columns='CLIMATE_CATEGORY',
                    values='OUTAGE_DURATION_MINS',
                    aggfunc='sum' )
prodname.assign(total=prodname.sum(axis=1)).sort_values('total', ascending=False).drop(columns='total')
prodname

CLIMATE_CATEGORY,cold,normal,warm
CAUSE_CATEGORY,,,
equipment failure,5240,89640,5050
fuel supply emergency,313794,130200,68399
intentional attack,58182,96034,19066
islanding,3889,2417,2518
public appeal,46770,46802,7751
severe weather,780629,1425825,680171
system operability disruption,21667,53638,14346


In [148]:
cause_category_byyear = outage_df.pivot_table(
                    index='CAUSE_CATEGORY', 
                    columns='YEAR',
                    values='OUTAGE_DURATION_MINS',
                    aggfunc='sum',
                    fill_value=0
                )
cause_category_byyear.assign(total=cause_category_byyear.sum(axis=1)).sort_values('total', ascending=False).drop(columns='total')
display_df(cause_category_byyear,
           rows=7,
           cols = 17)

YEAR,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016
CAUSE_CATEGORY,,,,,,,,,,,,,,,,,
equipment failure,0,494,0,3121,1342,435,159,1907,3242,86248,1653,54,105,1170,0,0,0
fuel supply emergency,0,0,0,0,0,0,0,1676,52735,13140,28980,49590,25950,45746,226177,255,68144
intentional attack,0,0,0,2659,0,0,0,0,0,0,0,64711,24931,27289,32694,8745,12253
islanding,0,0,0,0,0,0,0,47,1851,1521,999,1040,234,1679,56,1174,223
public appeal,0,769,0,1548,24915,0,714,4861,12115,3840,18922,26378,3062,0,2852,1347,0
severe weather,34050,10140,65901,188984,283638,284253,216815,115852,384949,175735,233357,332851,268295,131267,61334,82800,16404
system operability disruption,2910,6406,613,17700,289,915,2061,1837,5350,1376,27467,6948,2313,962,52,4875,7577


## Step 3: Assessment of Missingness

In [10]:
# TODO

## Step 4: Hypothesis Testing

In [11]:
# TODO

## Step 5: Framing a Prediction Problem

In [12]:
# TODO

## Step 6: Baseline Model

In [13]:
# TODO

## Step 7: Final Model

In [14]:
# TODO

## Step 8: Fairness Analysis

In [15]:
# TODO